# WIFUSDC：看看下载的真实数据

展示 CryptoHFTData 的盘口、逐笔成交、标记价。默认只读取 **2026-10-01 01:00–02:00 UTC（北京时间09:00–10:00）**，避免一次加载14天。

原始Parquet在项目 `data/cryptohftdata/raw/`。本笔记本只读文件，不联网、不修改数据。保存输出仅包含少量样例与图表；若不希望这些样例进入GitHub，提交前清除Notebook输出。

## 准备
在所选Jupyter内核安装依赖：`%pip install pyarrow pandas matplotlib`。修改下面的DATE/HOUR可以浏览其他小时。

In [ ]:
from pathlib import Path
from decimal import Decimal
import hashlib
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

# 从项目根目录或notebooks目录运行；其他位置请手动设置PROJECT_ROOT。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data/cryptohftdata/raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('请设置PROJECT_ROOT为移动硬盘上的cryptocurrency_quant项目目录')
DATE, HOUR = '2026-10-01', '01'
DATA_DIR = PROJECT_ROOT / 'data/cryptohftdata/raw/binance_futures' / DATE / HOUR
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 35)
plt.rcParams.update({'figure.figsize': (10, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('读取位置：', DATA_DIR)


## 1. 文件和原始字段
价格、数量保持原始字符串；绘图时另转float。时间字段保留整数，避免纳秒时间或更新编号被float舍入。

In [ ]:
tables = {kind: pq.read_table(DATA_DIR / f'WIFUSDC_{kind}.parquet')
          for kind in ['orderbook', 'trades', 'mark_price']}
frames = {kind: table.to_pandas(types_mapper=pd.ArrowDtype) for kind, table in tables.items()}
display(pd.DataFrame([{'类型': k, '行数': t.num_rows, '列数': t.num_columns,
                       '文件大小KB': round((DATA_DIR / f'WIFUSDC_{k}.parquet').stat().st_size / 1024, 1)}
                      for k, t in tables.items()]))
for kind, table in tables.items():
    print(kind)
    display(pd.DataFrame({'字段': table.schema.names, '原始类型': [str(f.type) for f in table.schema]}))


## 2. 盘口：快照与价位更新
`side=bid/ask`是买盘/卖盘；`price`是挂单价格；`quantity`是该价位**更新后的总挂单量**，不是新增量。数量为0表示删除价位。

`event_type=snapshot`用于初始化，`update`用于替换发生变化的价位。快照每一行只是一档，不能每读一行就清空盘口。`first_update_id/final_update_id/prev_final_update_id`对应币安U/u/pu；快照检查点是last_update_id。order_count为空不能解释成订单数0。

下面把同一买价的两条真实更新放在一起：数量变化也可能来自撤单，不能直接当成交量。

In [ ]:
book = frames['orderbook']
cols = ['event_type', 'side', 'price', 'quantity', 'event_time',
        'first_update_id', 'final_update_id', 'prev_final_update_id']
print('原始快照行（不是最优买价排序）：')
display(book.loc[book.event_type == 'snapshot', cols].head(5))
updates = book.loc[book.event_type == 'update'].copy()
updates['北京时间'] = pd.to_datetime(updates.event_time.astype('int64'), unit='ms', utc=True).dt.tz_convert('Asia/Shanghai')
example = updates.loc[(updates.side == 'bid') & (updates.price.map(Decimal) == Decimal('0.2481'))]
if len(example) < 2:
    keys = updates.groupby(['side', 'price'], sort=False).size()
    side, price = keys[keys >= 2].index[0]
    example = updates.loc[(updates.side == side) & (updates.price == price)]
display(example[['北京时间', *cols]].head(2))


### 同一条盘口消息会展开成多行
下面按相邻原生元数据划分消息组，展示首组价位更新。此规则用于本小时浏览，不能未经校验推广到所有历史。行数不是盘口消息数。

In [ ]:
group_fields = ['received_time', 'event_time', 'final_update_id']
key_rows = list(zip(*(updates[col].astype('string').fillna('<null>') for col in group_fields)))
group_ids, group_id, previous = [], -1, None
for key in key_rows:
    if key != previous:
        group_id += 1
    group_ids.append(group_id)
    previous = key
updates['消息组'] = group_ids
display(updates.loc[updates['消息组'] == 0, cols].head(10))
print(f'本小时更新价位行数：{len(updates):,}；按此规则分组：{updates["消息组"].nunique():,}组')


### 初始快照的最优十档
按价格排序展示买卖盘。图是小时起点的一个快照，不是整个小时的平均盘口；没有在这里回放全部增量。

In [ ]:
snap = book.loc[book.event_type == 'snapshot'].copy()
first_received = snap.received_time.iloc[0]
snap = snap.loc[snap.received_time == first_received].copy()
snap['px'] = snap.price.map(float)
snap['qty'] = snap.quantity.map(float)
bids = snap.loc[snap.side == 'bid'].sort_values('px', ascending=False).head(10)
asks = snap.loc[snap.side == 'ask'].sort_values('px').head(10)
display(pd.DataFrame({'买价': bids.px.to_numpy(), '买量WIF': bids.qty.to_numpy(),
                      '卖价': asks.px.to_numpy(), '卖量WIF': asks.qty.to_numpy()}))
fig, ax = plt.subplots()
ax.bar(bids.px, bids.qty, width=0.00007, color='#24756b', label='Bid')
ax.bar(asks.px, asks.qty, width=0.00007, color='#b96043', hatch='//', label='Ask')
ax.set(title=f'WIFUSDC initial snapshot | {DATE} {HOUR}:00 UTC', xlabel='Price (USDC/WIF)', ylabel='Resting quantity (WIF)')
ax.ticklabel_format(axis='x', style='plain', useOffset=False)
ax.legend(); fig.tight_layout(); plt.show()


## 3. 逐笔成交：实际成交的价格和数量
trade_id是成交编号，price/quantity是成交价/成交量；trade_time是成交发生时间，event_time是事件时间（毫秒），received_time是供应商接收时间（纳秒）。

is_buyer_maker=true表示买方是被动方，即主动卖出；false表示主动买入。成交价是已经发生的交易，不代表此刻可以买入/卖出的报价。

In [ ]:
trades = frames['trades'].copy()
trades['北京时间'] = pd.to_datetime(trades.trade_time.astype('int64'), unit='ms', utc=True).dt.tz_convert('Asia/Shanghai')
trades['主动方向'] = trades.is_buyer_maker.map({True: '主动卖出', False: '主动买入'})
display(trades[['trade_id', '北京时间', 'price', 'quantity', '主动方向', 'order_type']].head(8))
print('前8笔成交量的精确加总（WIF）：', sum(trades.quantity.head(8).map(Decimal)))


### 这一小时的成交价和逐分钟成交笔数
一个trade_id记一笔，不把平台行数与旧采集aggTrade条数直接比较。价格折线只是连接观测点，成交空窗不能理解为持续报价。

In [ ]:
ordered = trades.sort_values(['trade_time', 'trade_id'])
trade_times = pd.to_datetime(ordered.trade_time.astype('int64'), unit='ms', utc=True)
start = pd.Timestamp(f'{DATE} {HOUR}:00:00', tz='UTC')
minutes = pd.date_range(start, periods=60, freq='min')
counts = pd.Series(1, index=trade_times).resample('1min').sum().reindex(minutes, fill_value=0)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(trade_times, ordered.price.map(float), color='#24756b', linewidth=1, marker='.', markersize=2)
axes[0].set(title=f'WIFUSDC trades | {DATE} {HOUR}:00–{int(HOUR)+1:02d}:00 UTC', ylabel='Trade price (USDC/WIF)')
axes[1].bar(counts.index, counts.values, width=0.8/1440, color='#596579')
axes[1].set(ylabel='Trades / minute', xlabel='UTC time')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=__import__('datetime').timezone.utc))
fig.tight_layout(); plt.show()


## 4. 标记价：估值参考，不是可执行报价
mark_price是合约标记价；index_price是指数价；estimated_settle_price是预估结算参考；funding_rate是该消息中的资金费率，0.00005换成百分数为0.005%；next_funding_time是下一次资金费时间（毫秒）。实际结算仍需资金费结算记录。

它辅助理解合约估值和风险，不能代替盘口或逐笔成交。

In [ ]:
marks = frames['mark_price'].copy()
marks['北京时间'] = pd.to_datetime(marks.event_time.astype('int64'), unit='ms', utc=True).dt.tz_convert('Asia/Shanghai')
marks['资金费率%'] = marks.funding_rate.map(float) * 100
marks['下次资金费北京时间'] = pd.to_datetime(marks.next_funding_time.astype('int64'), unit='ms', utc=True).dt.tz_convert('Asia/Shanghai')
display(marks[['北京时间', 'mark_price', 'index_price', '资金费率%', '下次资金费北京时间']].head(5))
fig, ax = plt.subplots()
mark_times = pd.to_datetime(marks.event_time.astype('int64'), unit='ms', utc=True)
ax.plot(mark_times, marks.mark_price.map(float), label='Mark price', color='#596579')
ax.plot(mark_times, marks.index_price.map(float), label='Index price', color='#b96043', linestyle='--')
ax.set(title=f'WIFUSDC reference prices | {DATE} {HOUR}:00 UTC hour', xlabel='UTC time', ylabel='Price (USDC/WIF)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=__import__('datetime').timezone.utc))
ax.legend(); fig.tight_layout(); plt.show()


## 5. 检查与阅读边界
下方确认币对、成交编号重复和价格/数量范围。它只是样本浏览，不能证明14天完整，也未验收所有盘口编号连续性或队列模型。

快照的received_time可能是平台初始化时间，不能把它直接用于实测网络延迟。平台纳秒存储单位不代表纳秒级采集精度。

In [ ]:
for kind, frame in frames.items():
    assert set(frame.symbol) == {'WIFUSDC'}, kind
assert not trades.trade_id.duplicated().any(), '本小时存在重复成交ID，需要核查'
assert all(Decimal(x) > 0 for x in trades.price)
assert all(Decimal(x) >= 0 for x in trades.quantity)
print('本小时基础检查通过；尚未执行完整盘口连续性与回测可用性验收。')


## 下一步
修改DATE/HOUR浏览其他小时；进行做市回测前还需要按完整快照/消息组回放盘口，检查断链，再转换为HftBacktest事件格式。

字段口径与差异详见 `doc/CryptoHFTData数据与回测对接说明.md`；本文使用真实已下载文件，没有模拟数据。